# **1. Perkenalan Dataset**

Dataset yang digunakan adalah **Iris Dataset** dari `sklearn.datasets`.

- **Sumber**: Scikit-learn built-in dataset (UCI ML Repository)
- **Jumlah data**: 150 baris, 4 fitur
- **Target**: 3 kelas (Setosa, Versicolor, Virginica)
- **Task**: Multi-class Classification
- **Alasan pemilihan**: Dataset bersih (0 missing values), ringan, dan ideal untuk eksperimen klasifikasi.


# **2. Import Library**

Mengimpor pustaka Python yang dibutuhkan untuk analisis data dan pembangunan model machine learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully!')

# **3. Memuat Dataset**

Memuat Iris Dataset menggunakan `sklearn.datasets` dan mengkonversinya ke DataFrame pandas.

In [ ]:
# Load Iris Dataset
iris = load_iris()
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['target'] = iris.target
df['species'] = df['target'].map({0: 'setosa', 1: 'versicolor', 2: 'virginica'})

print('Dataset Shape:', df.shape)
print('\nFirst 5 rows:')
df.head()

# **4. Exploratory Data Analysis (EDA)**

Melakukan eksplorasi data untuk memahami karakteristik dataset, distribusi fitur, dan hubungan antar variabel.

In [ ]:
# 4.1 Informasi dasar dataset
print('=== Dataset Info ===')
print(df.info())
print('\n=== Statistik Deskriptif ===')
print(df.describe())
print('\n=== Missing Values ===')
print(df.isnull().sum())

In [ ]:
# 4.2 Distribusi kelas target
plt.figure(figsize=(8, 5))
sns.countplot(x='species', data=df, palette='Set2')
plt.title('Distribusi Kelas Target (Species)', fontsize=14)
plt.xlabel('Species')
plt.ylabel('Jumlah')
plt.tight_layout()
plt.show()
print('Distribusi target:', df['target'].value_counts().to_dict())

In [ ]:
# 4.3 Heatmap korelasi
plt.figure(figsize=(10, 7))
numeric_df = df.select_dtypes(include=np.number)
sns.heatmap(numeric_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Heatmap Korelasi Fitur', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 4.4 Distribusi fitur per kelas (pairplot)
feature_cols = iris.feature_names
sns.pairplot(df, vars=feature_cols, hue='species', palette='Set2')
plt.suptitle('Pairplot Fitur per Kelas', y=1.02, fontsize=14)
plt.show()

In [ ]:
# 4.5 Boxplot distribusi fitur
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for idx, col in enumerate(feature_cols):
    ax = axes[idx//2][idx%2]
    df.boxplot(column=col, by='species', ax=ax)
    ax.set_title(col)
    ax.set_xlabel('Species')
plt.suptitle('Boxplot Fitur per Kelas')
plt.tight_layout()
plt.show()

# **5. Data Preprocessing**

Preprocessing data mencakup: pengecekan missing values, normalisasi fitur, dan pembagian data train/test.

In [ ]:
# 5.1 Cek dan handle missing values (preventif)
imputer = SimpleImputer(strategy='median')
feature_cols = [c for c in df.columns if c not in ['target', 'species']]
df[feature_cols] = imputer.fit_transform(df[feature_cols])
print('Missing values setelah impute:', df[feature_cols].isnull().sum().sum())

In [ ]:
# 5.2 Cek dan hapus duplikat
print('Duplikat sebelum:', df.duplicated().sum())
df = df.drop_duplicates()
print('Duplikat setelah:', df.duplicated().sum())
print('Shape setelah drop duplikat:', df.shape)

In [ ]:
# 5.3 Normalisasi fitur dengan StandardScaler
X = df[feature_cols]
y = df['target']

scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_cols)

print('Statistik setelah scaling:')
print(X_scaled.describe().round(3))

In [ ]:
# 5.4 Deteksi outlier dengan IQR
Q1 = X_scaled.quantile(0.25)
Q3 = X_scaled.quantile(0.75)
IQR = Q3 - Q1
outlier_mask = ((X_scaled < (Q1 - 1.5 * IQR)) | (X_scaled > (Q3 + 1.5 * IQR))).any(axis=1)
print(f'Jumlah outlier terdeteksi: {outlier_mask.sum()} baris')
# Untuk iris kita pertahankan outlier karena dataset sudah sangat kecil
print('Outlier dipertahankan karena jumlah data terbatas.')

In [ ]:
# 5.5 Split train/test dan simpan hasil preprocessing
import os
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)

os.makedirs('nadataset_preprocessing', exist_ok=True)
X_train.to_csv('nadataset_preprocessing/X_train.csv', index=False)
X_test.to_csv('nadataset_preprocessing/X_test.csv', index=False)
y_train.to_csv('nadataset_preprocessing/y_train.csv', index=False)
y_test.to_csv('nadataset_preprocessing/y_test.csv', index=False)

print(f'Preprocessing selesai!')
print(f'Train set : {X_train.shape}')
print(f'Test set  : {X_test.shape}')
print(f'File tersimpan di folder nadataset_preprocessing/')